# Lab Tùy chọn: Kỹ thuật đặc trưng và Hồi quy đa thức

![](./images/C1_W2_Lab07_FeatureEngLecture.PNG)


## Mục tiêu
Trong lab này, bạn sẽ:
- khám phá kỹ thuật đặc trưng (feature engineering) và hồi quy đa thức, cho phép bạn sử dụng cơ chế của hồi quy tuyến tính để khớp các hàm rất phức tạp, thậm chí rất phi tuyến.


## Công cụ
Bạn sẽ sử dụng hàm đã xây dựng ở các lab trước cũng như matplotlib và NumPy. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import zscore_normalize_features, run_gradient_descent_feng
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays

<a name='FeatureEng'></a>
# Tổng quan về Kỹ thuật đặc trưng và Hồi quy đa thức

Về cơ bản, hồi quy tuyến tính cung cấp một phương tiện để xây dựng các mô hình có dạng:
$$f_{\mathbf{w},b} = w_0x_0 + w_1x_1+ ... + w_{n-1}x_{n-1} + b \tag{1}$$ 
Điều gì sẽ xảy ra nếu các đặc trưng/dữ liệu của bạn là phi tuyến hoặc là sự kết hợp của các đặc trưng? Ví dụ, giá nhà thường không tuyến tính với diện tích sử dụng mà lại phạt (penalize) những căn nhà quá nhỏ hoặc quá lớn, tạo ra các đường cong như trong hình minh họa ở trên. Làm sao chúng ta có thể sử dụng cơ chế của hồi quy tuyến tính để khớp đường cong này? Hãy nhớ rằng, 'cơ chế' mà chúng ta có là khả năng điều chỉnh các tham số $\mathbf{w}$, $\mathbf{b}$ trong (1) để 'khớp' phương trình với dữ liệu huấn luyện. Tuy nhiên, dù điều chỉnh $\mathbf{w}$,$\mathbf{b}$ trong (1) như thế nào cũng không thể khớp được với một đường cong phi tuyến.


<a name='PolynomialFeatures'></a>
## Đặc trưng đa thức (Polynomial Features)

Ở trên, chúng ta đang xem xét một tình huống mà dữ liệu là phi tuyến. Hãy thử sử dụng những gì chúng ta đã biết để khớp một đường cong phi tuyến. Chúng ta sẽ bắt đầu với một hàm bậc hai đơn giản: $y = 1+x^2$

Bạn đã quen thuộc với tất cả các hàm mà chúng ta đang sử dụng. Chúng có sẵn trong tệp lab_utils.py để tham khảo. Chúng ta sẽ sử dụng [`np.c_[..]`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html), một hàm của NumPy dùng để nối các mảng theo cột.

In [ ]:
# tạo dữ liệu mục tiêu
x = np.arange(0, 20, 1)
y = 1 + x**2
X = x.reshape(-1, 1)

model_w,model_b = run_gradient_descent_feng(X,y,iterations=1000, alpha = 1e-2)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("no feature engineering")
plt.plot(x,X@model_w + model_b, label="Predicted Value");  plt.xlabel("X"); plt.ylabel("y"); plt.legend(); plt.show()

Đúng như dự đoán, kết quả khớp không tốt. Điều cần thiết là một dạng như $y= w_0x_0^2 + b$, hay còn gọi là **đặc trưng đa thức**.
Để làm được điều này, bạn có thể chỉnh sửa *dữ liệu đầu vào* để *tạo ra (engineer)* các đặc trưng cần thiết. Nếu bạn thay dữ liệu gốc bằng phiên bản đã bình phương giá trị $x$, thì bạn có thể đạt được $y= w_0x_0^2 + b$. Hãy thử nhé. Thay `X` bằng `X**2` bên dưới:

In [ ]:
# tạo dữ liệu mục tiêu
x = np.arange(0, 20, 1)
y = 1 + x**2

# Tạo đặc trưng 
X = x**2      #<-- added engineered feature

In [ ]:
X = X.reshape(-1, 1)  #X phải là một ma trận 2 chiều
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha = 1e-5)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Added x**2 feature")
plt.plot(x, np.dot(X,model_w) + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

Tuyệt! gần như khớp hoàn hảo. Hãy chú ý giá trị của $\mathbf{w}$ và b được in ngay phía trên đồ thị: `w,b found by gradient descent: w: [1.], b: 0.0490`. Gradient descent đã điều chỉnh các giá trị ban đầu của $\mathbf{w},b $ thành (1.0,0.049) hay mô hình $y=1*x_0^2+0.049$, rất gần với mục tiêu $y=1*x_0^2+1$ của chúng ta. Nếu bạn chạy lâu hơn, kết quả có thể khớp tốt hơn nữa. 

### Lựa chọn đặc trưng
<a name='GDF'></a>
Ở trên, chúng ta đã biết trước rằng cần có một số hạng $x^2$. Không phải lúc nào cũng rõ ràng đặc trưng nào là cần thiết. Ta có thể thêm nhiều đặc trưng tiềm năng khác nhau để thử tìm ra những đặc trưng hữu ích nhất. Ví dụ, điều gì sẽ xảy ra nếu thay vào đó ta thử: $y=w_0x_0 + w_1x_1^2 + w_2x_2^3+b$ ? 

Hãy chạy các ô lệnh tiếp theo. 

In [ ]:
# tạo dữ liệu mục tiêu
x = np.arange(0, 20, 1)
y = x**2

# tạo đặc trưng .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature

In [ ]:
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha=1e-7)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("x, x**2, x**3 features")
plt.plot(x, X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

Lưu ý giá trị của $\mathbf{w}$, `[0.08 0.54 0.03]` và b là `0.0106`. Điều này ngụ ý mô hình sau khi khớp/huấn luyện là:
$$ 0.08x + 0.54x^2 + 0.03x^3 + 0.0106 $$
Gradient descent đã nhấn mạnh vào dữ liệu khớp tốt nhất với dữ liệu $x^2$ bằng cách tăng số hạng $w_1$ so với các số hạng còn lại. Nếu bạn chạy trong một thời gian rất dài, nó sẽ tiếp tục giảm ảnh hưởng của các số hạng khác. 
>Gradient descent tự chọn ra các đặc trưng 'đúng' cho chúng ta bằng cách nhấn mạnh tham số tương ứng của chúng

Hãy cùng xem lại ý tưởng này:
- trọng số nhỏ hơn ngụ ý đặc trưng ít quan trọng/ít đúng hơn, và trong trường hợp cực đoan, khi trọng số bằng 0 hoặc rất gần 0, đặc trưng tương ứng không hữu ích trong việc khớp mô hình với dữ liệu.
- ở trên, sau khi khớp, trọng số liên kết với đặc trưng $x^2$ lớn hơn nhiều so với trọng số của $x$ hay $x^3$ vì nó hữu ích nhất trong việc khớp dữ liệu. 

### Một góc nhìn khác
Ở trên, các đặc trưng đa thức được chọn dựa trên mức độ khớp của chúng với dữ liệu mục tiêu. Một cách khác để suy nghĩ về điều này là lưu ý rằng chúng ta vẫn đang sử dụng hồi quy tuyến tính sau khi đã tạo ra các đặc trưng mới. Với điều đó, các đặc trưng tốt nhất sẽ có quan hệ tuyến tính với biến mục tiêu. Điều này được hiểu rõ nhất qua một ví dụ. 

In [ ]:
# tạo dữ liệu mục tiêu
x = np.arange(0, 20, 1)
y = x**2

# tạo đặc trưng .
X = np.c_[x, x**2, x**3]   #<-- thêm đặc trưng đã tạo
X_features = ['x','x^2','x^3']

In [ ]:
fig,ax=plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X[:,i],y)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("y")
plt.show()

Ở trên, có thể thấy rõ đặc trưng $x^2$ khi vẽ theo giá trị mục tiêu $y$ có quan hệ tuyến tính. Hồi quy tuyến tính khi đó có thể dễ dàng tạo ra một mô hình sử dụng đặc trưng đó.

### Chuẩn hóa đặc trưng
Như đã mô tả trong lab trước, nếu tập dữ liệu có các đặc trưng với thang đo khác nhau đáng kể, ta nên áp dụng chuẩn hóa đặc trưng để tăng tốc gradient descent. Trong ví dụ trên, có $x$, $x^2$ và $x^3$ sẽ tự nhiên có thang đo rất khác nhau. Hãy áp dụng chuẩn hóa Z-score cho ví dụ của chúng ta.

In [ ]:
# tạo dữ liệu mục tiêu
x = np.arange(0,20,1)
X = np.c_[x, x**2, x**3]
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X,axis=0)}")

# thêm chuẩn hóa trung bình (mean normalization) 
X = zscore_normalize_features(X)     
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X,axis=0)}")

Bây giờ chúng ta có thể thử lại với một giá trị alpha mạnh hơn:

In [ ]:
x = np.arange(0,20,1)
y = x**2

X = np.c_[x, x**2, x**3]
X = zscore_normalize_features(X) 

model_w, model_b = run_gradient_descent_feng(X, y, iterations=100000, alpha=1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

Chuẩn hóa đặc trưng giúp quá trình này hội tụ nhanh hơn nhiều.   
Hãy chú ý lại giá trị của $\mathbf{w}$. Số hạng $w_1$, chính là số hạng $x^2$, được nhấn mạnh nhiều nhất. Gradient descent gần như đã loại bỏ hoàn toàn số hạng $x^3$.

### Các hàm phức tạp
Với kỹ thuật đặc trưng, ngay cả những hàm khá phức tạp cũng có thể được mô hình hóa:

In [ ]:
x = np.arange(0,20,1)
y = np.cos(x/2)

X = np.c_[x, x**2, x**3,x**4, x**5, x**6, x**7, x**8, x**9, x**10, x**11, x**12, x**13]
X = zscore_normalize_features(X) 

model_w,model_b = run_gradient_descent_feng(X, y, iterations=1000000, alpha = 1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()



## Chúc mừng!
Trong lab này, bạn đã:
- học được cách hồi quy tuyến tính có thể mô hình hóa các hàm phức tạp, thậm chí rất phi tuyến bằng kỹ thuật đặc trưng
- nhận ra rằng việc áp dụng chuẩn hóa đặc trưng là quan trọng khi thực hiện kỹ thuật đặc trưng